In [ ]:
import pandas as pd
import json
import re

base_path = ""

def clean_and_parse_result(raw):
    try:
        cleaned = raw.strip('"').replace('\\"', '"')
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {}

def load_and_parse(files, atc_code):
    dfs = []
    for f in files:
        df = pd.read_csv(base_path + f)

        # Parse 'result' JSON
        parsed_results = df["result"].apply(clean_and_parse_result)
        result_df = pd.json_normalize(parsed_results)

        # Drop old 'result' and add parsed columns + ATC code tag
        df = df.drop(columns=["result"]).reset_index(drop=True)
        df = pd.concat([df, result_df], axis=1)
        df["atc_code"] = atc_code

        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

# Load and tag each group
a10_df = load_and_parse(["a10_switch_stops.csv"], "a10")

a10_original = pd.read_csv("a10_switches_original.csv").dropna(subset=['anamnesis'])

def combine_dfs(original, df):
    combined = pd.concat([original.reset_index(drop=True), df.drop(columns=['id']).reset_index(drop=True)], axis=1)
    return combined

# Combine each pair
combined_a10 = combine_dfs(a10_original, a10_df)

# Concatenate all together
final_combined_df = combined_a10

# Optional: Check the result
print(final_combined_df.head())

unique_anamnesis_df = final_combined_df.drop_duplicates(subset='anamnesis')

reason_df = unique_anamnesis_df[unique_anamnesis_df['reason_for_stopping'] != ""]
print(len(reason_df))


In [ ]:
diabetes_drugs = [
    "metformin",
    "metformiin",
    "metformiin",
    "metformin",
    "metformiini",
    "metformini",
    "metformini",
    "metformin",
    "metforali",
    "metforal",
    "victoza",
    "diaprel",
    "jardiance",
    "amaryl",
    "janumet",
    "trajenta",
    "forxiga",
    "toujeo",
    "lantus",
    "levemir",
    "januvia",
    "insuliin",
    "insuliini",
    "insuliinpump",
    "xigduo",
    "jentadueto",
    "gliklada",
    "victozat",
    "diapreel",
    "jardiance",
    "victoza",
    "metforminum",
    "sglt-2",
    "insuliinravi",
    "gliklada",
    "victoza",
    "amaryl",
    "metformiin ja jardiance",
    "t.diaprel",
    "diabeedi",
    "gliclada",
    "humalog",
    "novomix",
    "novorapid",
    "tresiba",
    "nrapid",
    "metformi",
    "biguanidi",
    "glimeperid",
    "gliclada",
    "glicladat",
    "gliclazidi",
    "januuvia",
    "saksagliptiin",
    "jardians",
    "synjardi",
    "jardiace",
    "pioglitazon",
    "comboglyze2.5/1.0"
]


# Make sure all terms are lowercase
diabetes_drugs = [drug.lower() for drug in diabetes_drugs]

# Join the list into a single regex pattern (escaped, joined with | for "OR" logic)
pattern = "|".join(re.escape(drug) for drug in diabetes_drugs)

# Filter drug names that contain any diabetes-related term (case-insensitive)
diabetes_df = reason_df[reason_df['drug_name'].str.lower().str.contains(pattern, na=False)]



In [ ]:
from vllm import LLM, SamplingParams
from vllm.sampling_params import GuidedDecodingParams
from transformers import AutoTokenizer
import torch
from openai import OpenAI

import json
import csv
import re
import math
import numpy as np
import pandas as pd
import asyncio
import matplotlib.pyplot as plt 

from pydantic import BaseModel
from enum import Enum

from huggingface_hub import login
HF_TOKEN = ""
login(HF_TOKEN)  # This logs you in for the session


import openai
openai.api_type = "azure"

with open('azure_api', 'r') as api_file:
    openai.api_key=str(api_file.readline()).strip()




# llm = LLM(model=model_name, device=device, max_model_len=65536, tensor_parallel_size=1, enable_prefix_caching=True)
# tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
print(len(diabetes_df))

In [ ]:
from openai import AzureOpenAI

client = AzureOpenAI(
    api_key=openai.api_key,
    azure_endpoint="",
    api_version=''
)
model_name = "gpt-4o"
deployment = "gpt-4o"

In [ ]:
# Define your classification categories

categories = [
    "Adverse reactions",
    "Treatment success", #
    "Treatment inefficacy", #
    "Contraindication", #
    "Non-medical reasons",
    "Other",
    "Indeterminate"
]

example_reasons = [
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    ""
    
]

example_labels = [
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    ""
]


# Set up guided decoding to restrict outputs to your categories
guided_decoding = GuidedDecodingParams(choice=categories)

# Configure sampling parameters with guided decoding
sampling_params = SamplingParams(guided_decoding=guided_decoding)


system_prompt = {
"role": "system", "content":"""You are a helpful assistant that reads Estonian electronic health records written by doctors about diabetes medication discontinuation for their patients with high blood sugar levels. For each input, classify the reason why the patient stopped taking the medication.
Your input is the extracted reason for stopping the drug.

Here are the categories you must use:

1. Adverse reactions: Discontinuation due to adverse side effects, allergic reactions, or negative interactions with other medications.
2. Treatment success: Discontinuation due to successful treatment completion or sufficient improvement in health.
3. Treatment inefficacy: Perceived ineffectiveness of the treatment or loss of belief in its efficacy.
4. Contraindication: Discontinuation due to the emergence or discovery of a medical condition or risk factor that makes the continued use of the treatment unsafe or inappropriate (e.g. "vastunäidustatud").
5. Non-medical reasons: Discontinuation due to factors unrelated to the patient's health or the treatment’s medical effects, such as financial constraints, access issues, personal choice, cultural beliefs, or social circumstances.
6. Other: Other medical reasons not explicitly covered by the other categories.
7. Indeterminate: Unclear or unspecified reasons for discontinuation.

Only choose one category per input. If the reason is ambiguous or not clearly stated, select "Indeterminate". Respond only with the category name."""}


messages_template = [system_prompt]

# drug_stop_list = diabetes_df['reason_for_stopping'].tolist()
diabetes_df['classification'] = ""

# drug_stop_list = drug_stop_list[:10]
# anamnesis_list = anamnesis_list[:10]

for reason, label in zip(example_reasons, example_labels):
    user_content = f"Reason for stopping:\n{reason}"
    messages_template.append({"role": "user", "content": user_content})
    messages_template.append({"role": "assistant", "content": label})


# Create the prompt for classification

reasons = []

for index, row in diabetes_df.iterrows():
    reason = row['reason_for_stopping']
    # Construct the message sequence
    messages = messages_template + [{"role": "user", "content": f"Reason for stopping:\n{reason}"}]
    try:
        # Make the API call
        response = client.chat.completions.create(
            model="gpt-4o",  # Replace with your deployed model name
            messages=messages,
            temperature=0,
            max_tokens=10,
            top_p=1.0,
            frequency_penalty=0,
            presence_penalty=0
        )
        # Extract the classification result
        classification = response.choices[0].message.content.strip()
    except Exception as e:
        # Handle any exceptions by assigning 'Error'
        classification = "Error"
    # Update the DataFrame with the classification result
    diabetes_df.at[index, 'classification'] = classification
# Extract and print the classification result
# classification = outputs[0].outputs[0].text.strip()

In [ ]:
diabetes_df.to_csv('diabetes_df_gpt_classifications')

In [ ]:
# classified_categories = [output.outputs[0].text.strip() for output in outputs]

# Build a dataframe
df = pd.DataFrame({
    "Input": drug_stop_list,
    "Category": gpt_reasons
})

# Save to CSV
df.to_csv("a10_switches_why_stopped_gpt.csv", index=False)

# Optional: print path or success message
print("Results saved to a10_switches_why_stopped_gpt.csv")